# Faza 3 - trening wariantu D (12 runów: 4 modele × 3 seedy)

Realne (30/klasę) + syntetyczne (120/klasę z `manifest_kept.csv`), bez RandAugment. Porównujemy do A (no aug) i B (RandAug) z fazy 2.

## 1. Klonowanie + install

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Pets + HF login + pull `variant_D/`

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py

In [ ]:
from huggingface_hub import login, snapshot_download, whoami

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('brak HF_TOKEN - ustaw w Colab Secrets')
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

HF_REPO_ID = 'micwuj/dlicv-synth'
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    allow_patterns='variant_D/**',
    local_dir='data/synthetic',
)

## 3. Sanity check

In [ ]:
import json

synth_root = Path('data/synthetic/variant_D')
manifest = synth_root / 'manifest_kept.csv'
decision = synth_root / 'decision_v2.json'
assert manifest.exists(), f'brak manifestu: {manifest}'
assert decision.exists(), f'brak decision_v2.json'

print('decision_v2.json:')
print(json.dumps(json.loads(decision.read_text()), indent=2))

import csv
rows = list(csv.DictReader(open(manifest)))
from collections import Counter
by_breed = Counter(r['breed'] for r in rows)
print(f'\nmanifest: {len(rows)} kept rows across {len(by_breed)} breeds')
for b, n in sorted(by_breed.items()):
    print(f'  {b}: {n}')

!python scripts/generate_configs.py | tail -15
d_configs = sorted(Path('configs/exp').glob('D_*.yaml'))
print(f'\nD configs: {len(d_configs)} files')

## 4. GPU check

In [ ]:
from src.utils.device import device_info, get_device
print('device:', device_info(get_device()))

## 5. Dry-run (co zostanie odpalone)

In [ ]:
!python scripts/run_all.py --pattern 'D_*.yaml' --overrides configs/colab.yaml --skip-existing --dry-run

## 6. Smoke test - 1 run

Zanim odpalisz cały batch, puść jeden run dla pewności że dataset wrapper i manifest działają na Colab.

In [ ]:
!python -m src.train --config configs/base.yaml configs/colab.yaml configs/exp/D_resnet18_seed0.yaml

## 7. Pełny batch (12 runów)

`--skip-existing` pomija D_resnet18_seed0 (już zrobione w smoke teście).

In [ ]:
!python scripts/run_all.py --pattern 'D_*.yaml' --overrides configs/colab.yaml --skip-existing

## 8. Agregacja + porównanie A/B/D

In [ ]:
!python scripts/aggregate_results.py

In [ ]:
import pandas as pd
df = pd.read_csv('outputs/_aggregate/summary_by_model_variant.csv')
df